In [27]:
# Library

import os 
from getpass import getpass
from dotenv import load_dotenv
from IPython.display import display, Markdown

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

True

In [2]:
# must enter API key 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# below should not be changed
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# you can change this as preferred
os.environ["LANGCHAIN_PROJECT"] = "damar-ananta-langsmith-openai"

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Basic Prompting

In [ ]:
""" 
We'll start by looking at the various parts of our prompt. For RAG use-cases we'll typically have three core components however this is very use-cases dependant and can vary significantly. Nonetheless, for RAG we will typically see:

- Rules for our LLM: this part of the prompt sets up the behavior of our LLM, how it should approach responding to user queries, and simply providing as much information as possible about what we're wanting to do as possible. We typically place this within the system prompt of an chat LLM.
- Context: this part is RAG-specific. The context refers to some external information that we may have retrieved from a web search, database query, or often a vector database. This external information is the Retrieval Augmentation part of RAG. For chat LLMs we'll typically place this inside the chat messages between the assistant and user.
- Question: this is the input from our user. In the vast majority of cases the question/query/user input will always be provided to the LLM (and typically through a user message). However, the format and location of this being provided often changes.
- Answer: this is the answer from our assistant, again this is very typical and we'd expect this with every use-case.
"""

In [ ]:
""" 
The below is an example of how a RAG prompt may look:

Answer the question based on the context below,                 }
if you cannot answer the question using the                     }--->  (Rules) For Our Prompt
provided information answer with "I don't know"                 }

Context: Aurelio AI is an AI development studio                 }
focused on the fields of Natural Language Processing (NLP)      }
and information retrieval using modern tooling                  }--->   Context AI has
such as Large Language Models (LLMs),                           }
vector databases, and LangChain.                                }

Question: Does Aurelio AI do anything related to LangChain?     }--->   User Question

Answer:                                                         }--->   AI Answer
"""

In [ ]:
# Rules
prompt = """ 
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Context: {context}
"""

In [9]:
# passing the template to the Langchain model 
prompt_template = ChatPromptTemplate.from_messages([
    ("system", prompt),
    ("user", "{query}")
])

In [ ]:
# We can see that these variables have been picked up by our template object by viewing it's input_variables attribute
prompt_template.input_variables

['context', 'query']

In [11]:
# We can also view the structure of the messages (currently prompt templates) that the ChatPromptTemplate will construct by viewing the messages attribute
prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]

In [14]:
# Using these tuples is shorthand for the following, more explicit code

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(prompt),
    HumanMessagePromptTemplate.from_template("{query}")
])

prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]

# Invoking our LLM with Templates

In [ ]:
openai_model = "gpt-4o-mini"

llm = ChatOpenAI(temperature=0.0, model=openai_model)

In [ ]:
""" 
The temperature parameter controls the randomness of the LLM's output. A temperature of 0.0 makes an LLM's output more determinstic which in theory should lead to a lower likelihood of hallucination.

Now, the question here may be, why would we ever not use temperature=0.0? The answer to that is that sometimes a little bit of randomness can useful. 
Randomness tends to translate to text that feels more human and creative, so if we'd like an LLM to help us write an article or even a poem, that lack of determinism becomes a feature rather than a bug.
"""

In [17]:
pipeline = (
    {
        "query": lambda x: x["query"],
        "context": lambda x: x["context"]
    }
    | prompt_template
    | llm
)

In [ ]:
# Context
context = """Aurelio AI is an AI company developing tooling for AI
engineers. Their focus is on language AI with the team having strong
expertise in building AI agents and a strong background in
information retrieval.

The company is behind several open source frameworks, most notably
Semantic Router and Semantic Chunkers. They also have an AI
Platform providing engineers with tooling to help them build with
AI. Finally, the team also provides development services to other
organizations to help them bring their AI tech to market.

Aurelio AI became LangChain Experts in September 2024 after a long
track record of delivering AI solutions built with the LangChain
ecosystem."""

# Question by user manual input
query = "what does Aurelio AI do?"

In [ ]:
pipeline.invoke({"query": query, "context": context})

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


AIMessage(content='Aurelio AI is an AI company that develops tooling for AI engineers, focusing on language AI. They have expertise in building AI agents and information retrieval. The company is known for several open source frameworks, including Semantic Router and Semantic Chunkers, and offers an AI Platform that provides engineers with tools to build with AI. Additionally, they provide development services to help other organizations bring their AI technology to market.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 183, 'total_tokens': 264, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_48eea167bf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20b9c-1099-7793-9c72-0bce29208d22-0

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

# Few Shot Prompting

In [ ]:
""" 
Secara sederhana, Few-Shot Prompting adalah teknik memberikan beberapa contoh (examples) kepada LLM sebelum meminta LLM mengerjakan suatu tugas.

Tujuannya agar LLM memahami:
1. Pola input yang diberikan.
2. Pola output yang diharapkan.
3. Format atau gaya jawaban yang harus diikuti.
Jadi, kita tidak hanya memberikan instruksi kepada LLM, tetapi juga memberikan contoh bagaimana instruksi tersebut harus dikerjakan.
"""

In [20]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

In [21]:
examples = [
    {"input": "Here is query #1", "output": "Here is the answer #1"},
    {"input": "Here is query #2", "output": "Here is the answer #2"},
    {"input": "Here is query #3", "output": "Here is the answer #3"},
]

In [24]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt= example_prompt,
    examples= examples
)

print(few_shot_prompt.format())

Human: Here is query #1
AI: Here is the answer #1
Human: Here is query #2
AI: Here is the answer #2
Human: Here is query #3
AI: Here is the answer #3


## Few Shot Example

In [25]:
new_system_prompt = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Always answer in markdown format. When doing so please
provide headers, short summaries, follow with bullet
points, then conclude.

Context: {context}
"""

prompt_template.messages[0].prompt.template = new_system_prompt

out = pipeline.invoke({"query": query, "context": context}).content
print(out)

# Overview of Aurelio AI

Aurelio AI is an AI company that specializes in developing tools and solutions for AI engineers, particularly focusing on language AI.

## Key Areas of Focus
- **Language AI Development**: Expertise in building AI agents and information retrieval systems.
- **Open Source Frameworks**: Creator of notable frameworks like Semantic Router and Semantic Chunkers.
- **AI Platform**: Provides tooling for engineers to facilitate AI development.
- **Development Services**: Offers services to help organizations bring their AI technologies to market.

## Achievements
- Became LangChain Experts in September 2024, showcasing their proficiency in the LangChain ecosystem.

In conclusion, Aurelio AI is dedicated to empowering AI engineers through innovative tools, frameworks, and expert services in the field of language AI.


In [28]:
display(Markdown(out))

# Overview of Aurelio AI

Aurelio AI is an AI company that specializes in developing tools and solutions for AI engineers, particularly focusing on language AI.

## Key Areas of Focus
- **Language AI Development**: Expertise in building AI agents and information retrieval systems.
- **Open Source Frameworks**: Creator of notable frameworks like Semantic Router and Semantic Chunkers.
- **AI Platform**: Provides tooling for engineers to facilitate AI development.
- **Development Services**: Offers services to help organizations bring their AI technologies to market.

## Achievements
- Became LangChain Experts in September 2024, showcasing their proficiency in the LangChain ecosystem.

In conclusion, Aurelio AI is dedicated to empowering AI engineers through innovative tools, frameworks, and expert services in the field of language AI.

In [ ]:
""" 
This is not bad, but also not quite the format we wanted. We could try improving our initial prompting instructions, but when this doesn't work we can move on to our few-shot prompting. We want to build something like this:

Answer the user's query based on the context below,                 }
if you cannot answer the question using the                         }
provided information answer with "I don't know"                     }
                                                                    }--->  (Rules)
Always answer in markdown format. When doing so please              }
provide headers, short summaries, follow with bullet                }
points, then conclude. Here are some examples:                      }


User: Can you explain gravity?                                      }
AI: ## Gravity                                                      }
                                                                    }
Gravity is one of the fundamental forces in the universe.           }
                                                                    }
### Discovery                                                       }--->  (Example 1)
                                                                    }
* Gravity was first discovered by...                                }
                                                                    }
**To conclude**, Gravity is a fascinating topic and has been...     }
                                                                    }

User: What is the capital of France?                                }
AI: ## France                                                       }
                                                                    }
The capital of France is Paris.                                     }
                                                                    }--->  (Example 2)
### Origins                                                         }
                                                                    }
* The name Paris comes from the...                                  }
                                                                    }
**To conclude**, Paris is highly regarded as one of the...          }

Context: {context}                                                  }--->  (Context)
"""

In [29]:
examples = [
    {
        "input": "Can you explain gravity?",
        "output": (
            "## Gravity\n\n"
            "Gravity is one of the fundamental forces in the universe.\n\n"
            "### Discovery\n\n"
            "* Gravity was first discovered by Sir Isaac Newton in the late 17th century.\n"
            "* It was said that Newton theorized about gravity after seeing an apple fall from a tree.\n\n"
            "### In General Relativity\n\n"
            "* Gravity is described as the curvature of spacetime.\n"
            "* The more massive an object is, the more it curves spacetime.\n"
            "* This curvature is what causes objects to fall towards each other.\n\n"
            "### Gravitons\n\n"
            "* Gravitons are hypothetical particles that mediate the force of gravity.\n"
            "* They have not yet been detected.\n\n"
            "**To conclude**, Gravity is a fascinating topic and has been studied extensively since the time of Newton.\n\n"
        )
    },
    {
        "input": "What is the capital of France?",
        "output": (
            "## France\n\n"
            "The capital of France is Paris.\n\n"
            "### Origins\n\n"
            "* The name Paris comes from the Latin word \"Parisini\" which referred to a Celtic people living in the area.\n"
            "* The Romans named the city Lutetia, which means \"the place where the river turns\".\n"
            "* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.\n\n"
            "**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world's greatest cultural and economic centres.\n\n"
        )
    }
]

In [31]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [33]:
out = few_shot_prompt.format()
display(Markdown(out))

Human: Can you explain gravity?
AI: ## Gravity

Gravity is one of the fundamental forces in the universe.

### Discovery

* Gravity was first discovered by Sir Isaac Newton in the late 17th century.
* It was said that Newton theorized about gravity after seeing an apple fall from a tree.

### In General Relativity

* Gravity is described as the curvature of spacetime.
* The more massive an object is, the more it curves spacetime.
* This curvature is what causes objects to fall towards each other.

### Gravitons

* Gravitons are hypothetical particles that mediate the force of gravity.
* They have not yet been detected.

**To conclude**, Gravity is a fascinating topic and has been studied extensively since the time of Newton.


Human: What is the capital of France?
AI: ## France

The capital of France is Paris.

### Origins

* The name Paris comes from the Latin word "Parisini" which referred to a Celtic people living in the area.
* The Romans named the city Lutetia, which means "the place where the river turns".
* The city was renamed Paris in the 3rd century BC by the Celtic-speaking Parisii tribe.

**To conclude**, Paris is highly regarded as one of the most beautiful cities in the world and is one of the world's greatest cultural and economic centres.



In [34]:
few_shot_prompt

FewShotChatMessagePromptTemplate(examples=[{'input': 'Can you explain gravity?', 'output': '## Gravity\n\nGravity is one of the fundamental forces in the universe.\n\n### Discovery\n\n* Gravity was first discovered by Sir Isaac Newton in the late 17th century.\n* It was said that Newton theorized about gravity after seeing an apple fall from a tree.\n\n### In General Relativity\n\n* Gravity is described as the curvature of spacetime.\n* The more massive an object is, the more it curves spacetime.\n* This curvature is what causes objects to fall towards each other.\n\n### Gravitons\n\n* Gravitons are hypothetical particles that mediate the force of gravity.\n* They have not yet been detected.\n\n**To conclude**, Gravity is a fascinating topic and has been studied extensively since the time of Newton.\n\n'}, {'input': 'What is the capital of France?', 'output': '## France\n\nThe capital of France is Paris.\n\n### Origins\n\n* The name Paris comes from the Latin word "Parisini" which refe

In [35]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", new_system_prompt),
    few_shot_prompt,
    ("user", "{query}")
])

pipeline = prompt_template | llm
out = pipeline.invoke({
    "query": query,
    "context": context
}).content

display(Markdown(out))

## Aurelio AI Overview

Aurelio AI is an AI company focused on developing tools and services for AI engineers, particularly in the realm of language AI.

### Key Areas of Focus

- **Language AI**: Specializes in creating AI solutions that understand and generate human language.
- **Open Source Frameworks**: Developed notable frameworks such as:
  - **Semantic Router**
  - **Semantic Chunkers**
- **AI Platform**: Provides engineers with tools to facilitate AI development.
- **Development Services**: Offers assistance to organizations in bringing their AI technologies to market.

### Expertise

- The team has strong expertise in building AI agents.
- They possess a solid background in information retrieval.

### Recognition

- Became LangChain Experts in September 2024, showcasing their proficiency in the LangChain ecosystem.

**To conclude**, Aurelio AI is dedicated to empowering AI engineers through innovative tools, frameworks, and development services in the language AI domain.

# Chain of Thought Prompting

In [ ]:
""" 
We'll take a look at one more commonly used prompting technique called chain of thought (CoT). CoT is a technique that encourages the LLM to think through the problem step by step before providing an answer. The idea being that by breaking down the problem into smaller steps, the LLM is more likely to arrive at the correct answer and we are less likely to see hallucinations.
To implement CoT we don't need any specific LangChain objects, instead we are simply modifying how we instruct our LLM within the system prompt. We will ask the LLM to list the problems that need to be solved, to solve each problem individually, and then to arrive at the final answer.
"""

In [36]:
no_cot_system_prompt = """ 
Be a helpful assistant and answer the user's question.

You MUST answer the question directly without any other text of explanation
"""

no_cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", no_cot_system_prompt),
    ("user", "{query}")
])

In [37]:
query = ("How many keystrokes are needed to type the numbers from 1 to 500")

no_cot_pipeline = no_cot_prompt_template | llm
no_cot_result = no_cot_pipeline.invoke({"query": query}).content
print(no_cot_result)

The total number of keystrokes needed to type the numbers from 1 to 500 is 1,500.


In [ ]:
""" 
The actual answer is 1392, but the LLM without CoT just hallucinates and gives us a guess. Now, we can add explicit CoT prompting to our system prompt to see if we can get a better result.
"""

In [38]:
# Define the chain-of-thought prompt template 
cot_system_prompt = """ 
Be a helpful assistant and answer the user's question.

To answer the question, you must:
- List systematically and in precise detail all subproblems that need to be solved to answer the question
- Solve each suh problem INDIVIDUALLY and in sequence.
- Finally, use everything you have worked through to provide the final anwer.
"""

cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", cot_system_prompt),
    ("user", "{query}")
])

cot_pipeline = cot_prompt_template | llm

cot_result = cot_pipeline.invoke({
    "query": query
}).content

display(Markdown(cot_result))

To determine how many keystrokes are needed to type the numbers from 1 to 500, we can break down the problem into several subproblems:

### Subproblems:
1. Count the number of digits in the numbers from 1 to 9.
2. Count the number of digits in the numbers from 10 to 99.
3. Count the number of digits in the numbers from 100 to 499.
4. Count the digits in the number 500.
5. Sum all the digits counted in the previous steps to get the total number of keystrokes.

### Step 1: Count the number of digits in the numbers from 1 to 9
- The numbers from 1 to 9 are single-digit numbers.
- There are 9 numbers (1, 2, 3, 4, 5, 6, 7, 8, 9).
- Each number contributes 1 keystroke.
- Total keystrokes for this range = 9 * 1 = 9.

### Step 2: Count the number of digits in the numbers from 10 to 99
- The numbers from 10 to 99 are two-digit numbers.
- There are 90 numbers (10 to 99 inclusive).
- Each number contributes 2 keystrokes.
- Total keystrokes for this range = 90 * 2 = 180.

### Step 3: Count the number of digits in the numbers from 100 to 499
- The numbers from 100 to 499 are three-digit numbers.
- There are 400 numbers (100 to 499 inclusive).
- Each number contributes 3 keystrokes.
- Total keystrokes for this range = 400 * 3 = 1200.

### Step 4: Count the digits in the number 500
- The number 500 is a three-digit number.
- It contributes 3 keystrokes.
- Total keystrokes for this number = 3.

### Step 5: Sum all the digits counted
- Total keystrokes = Keystrokes from 1 to 9 + Keystrokes from 10 to 99 + Keystrokes from 100 to 499 + Keystrokes from 500
- Total keystrokes = 9 + 180 + 1200 + 3 = 1392.

### Final Answer:
The total number of keystrokes needed to type the numbers from 1 to 500 is **1392**.

In [ ]:
""" 
Now we get a much better result! Our LLM provides us with a final answer of 1392 which is correct. Finally, as mentioned most LLMs are now trained to use CoT prompting by default. So let's see what happens if we don't explicitly tell the LLM to use CoT.
"""

In [41]:
system_prompt = """ 
Be a helpful assistant and answer the user's question.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{query}")
])

pipeline = prompt_template | llm
result = pipeline.invoke({"query":query}).content
display(Markdown(result))

To calculate the total number of keystrokes needed to type the numbers from 1 to 500, we can break it down by the number of digits in the numbers:

1. **Numbers from 1 to 9** (1-digit numbers):
   - There are 9 numbers (1 to 9).
   - Each number requires 1 keystroke.
   - Total keystrokes = 9 × 1 = 9.

2. **Numbers from 10 to 99** (2-digit numbers):
   - There are 90 numbers (10 to 99).
   - Each number requires 2 keystrokes.
   - Total keystrokes = 90 × 2 = 180.

3. **Numbers from 100 to 499** (3-digit numbers):
   - There are 400 numbers (100 to 499).
   - Each number requires 3 keystrokes.
   - Total keystrokes = 400 × 3 = 1200.

4. **Number 500** (3-digit number):
   - This number requires 3 keystrokes.
   - Total keystrokes = 3.

Now, we can sum all the keystrokes:

- From 1 to 9: 9 keystrokes
- From 10 to 99: 180 keystrokes
- From 100 to 499: 1200 keystrokes
- Number 500: 3 keystrokes

Total keystrokes = 9 + 180 + 1200 + 3 = 1392.

Therefore, the total number of keystrokes needed to type the numbers from 1 to 500 is **1392**.